# VADER Sentiment Analysis

This notebook applies VADER sentiment analysis to the cleaned product review dataset. It compares the VADER result with the original sentiment label already included in the dataset.

## 1. What VADER Does

VADER stands for Valence Aware Dictionary and sEntiment Reasoner. It is a rule-based sentiment analysis tool that is useful for short text such as reviews, comments, and social media posts.

VADER does not train a machine learning model in this project. Instead, it uses a built-in sentiment dictionary and rules to score text.

## 2. Import Libraries

Import pandas for data handling, matplotlib for charts, and VADER for sentiment scoring.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## 3. Load Cleaned Dataset

Load the cleaned dataset created during Stage 2.

In [ ]:
data_path = "../data/processed/cleaned_reviews.csv"
df = pd.read_csv(data_path)
df.head()

## 4. Understand VADER Scores

VADER returns four scores for each text:

- `pos`: how positive the text is
- `neu`: how neutral the text is
- `neg`: how negative the text is
- `compound`: one overall score from -1 to +1

A compound score closer to +1 is more positive. A score closer to -1 is more negative. A score near 0 is more neutral.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

example_text = df.loc[0, "full_review_text"]
print(example_text)
analyzer.polarity_scores(example_text)

## 5. Apply VADER to All Reviews

Apply VADER scoring to the `full_review_text` column and create separate score columns.

In [ ]:
scores = df["full_review_text"].fillna("").apply(analyzer.polarity_scores)
scores_df = pd.DataFrame(scores.tolist())

df["vader_negative"] = scores_df["neg"]
df["vader_neutral"] = scores_df["neu"]
df["vader_positive"] = scores_df["pos"]
df["vader_compound"] = scores_df["compound"]

df[["full_review_text", "vader_negative", "vader_neutral", "vader_positive", "vader_compound"]].head()

## 6. Convert Compound Score into Sentiment Labels

Use the common VADER thresholds:

- compound score >= 0.05 means positive
- compound score <= -0.05 means negative
- otherwise means neutral

In [ ]:
def label_vader_sentiment(compound_score):
    if compound_score >= 0.05:
        return "positive"
    if compound_score <= -0.05:
        return "negative"
    return "neutral"


df["vader_sentiment"] = df["vader_compound"].apply(label_vader_sentiment)
df[["sentiment", "vader_sentiment", "vader_compound"]].head()

## 7. Compare VADER with Dataset Sentiment

Create a `sentiment_match` column to check whether the original dataset sentiment label is the same as the VADER label.

In [ ]:
df["sentiment_match"] = df["sentiment"] == df["vader_sentiment"]

match_percentage = df["sentiment_match"].mean() * 100
print(f"Sentiment match percentage: {match_percentage:.2f}%")

## 8. Original Sentiment Distribution

Visualize the sentiment labels that came with the dataset.

In [ ]:
df["sentiment"].value_counts().plot(kind="bar", figsize=(8, 4), color="steelblue")
plt.title("Original Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.show()

## 9. VADER Sentiment Distribution

Visualize the sentiment labels created from the VADER compound score.

In [ ]:
df["vader_sentiment"].value_counts().plot(kind="bar", figsize=(8, 4), color="seagreen")
plt.title("VADER Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.show()

## 10. Rating Distribution

Check how product ratings are distributed in the cleaned dataset.

In [ ]:
df["rating"].value_counts().sort_index().plot(kind="bar", figsize=(8, 4), color="darkorange")
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.show()

## 11. Review Word Count Distribution

Review length can help explain sentiment quality. Very short reviews may be harder for any sentiment method to classify correctly.

In [ ]:
df["word_count"].plot(kind="hist", bins=50, figsize=(10, 4), color="mediumpurple")
plt.title("Review Word Count Distribution")
plt.xlabel("Word Count")
plt.ylabel("Number of Reviews")
plt.show()

## 12. Most Positive and Most Negative Reviews

View examples with the highest and lowest VADER compound scores.

In [ ]:
display(df.nlargest(5, "vader_compound")[["product_name", "rating", "sentiment", "vader_sentiment", "vader_compound", "full_review_text"]])
display(df.nsmallest(5, "vader_compound")[["product_name", "rating", "sentiment", "vader_sentiment", "vader_compound", "full_review_text"]])

## 13. Save Sentiment Results

Save the VADER scoring results for later project stages. This is still rule-based sentiment analysis, not machine learning model training.

In [ ]:
output_path = "../data/processed/sentiment_results.csv"
df.to_csv(output_path, index=False)
print(f"Saved sentiment results to: {output_path}")